# Vision Transformer (ViT) from Scratch

A minimal Vision Transformer encoder written in PyTorch, following [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929).

What's implemented by hand:
- **Patch embedding**: a 16x16 conv with stride 16 turns a 256x256 image into 256 patch tokens of size 768
- **Sinusoidal positional encoding**
- **Multi-head self-attention** (Q, K, V projections, head splitting, scaled dot product, output projection)
- **Pre-norm transformer block** with a GELU feed-forward network and residual connections
- **Learnable [CLS] token** prepended to the sequence

Input `(B, 3, 256, 256)` gives output `(B, 257, 768)`: 256 patch tokens plus the CLS token.

## Model

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class PatchEmbed(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    self.patch_model = nn.Conv2d(in_channels, 768, stride=16, kernel_size=16)

  def forward(self, x):
    patches = self.patch_model(x)  # B, 768, 16, 16
    patches = patches.flatten(2).permute(0, 2, 1)  # B, 256, 768
    return patches


class PositionalEmbedding(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    _, num_tokens, hdim = x.shape

    pos = torch.arange(num_tokens, device=x.device).unsqueeze(1)  # T, 1
    div = torch.exp(torch.arange(0, hdim, 2, device=x.device) * (-math.log(10000.0) / hdim))
    pe = torch.zeros(num_tokens, hdim, device=x.device)
    # print((pos*div).shape)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return x + pe.unsqueeze(0)


class FeedForwardNetwork(nn.Module):
  def __init__(self, hdim=768):
    super().__init__()
    self.projection = nn.Sequential(nn.Linear(hdim, 4 * hdim), nn.GELU(), nn.Linear(4 * hdim, hdim))

  def forward(self, x):
    return self.projection(x)


class TransformerLayer(nn.Module):
  def __init__(self, hdim=768, num_heads=4):
    super().__init__()
    self.Wq = nn.Linear(hdim, hdim)
    self.Wk = nn.Linear(hdim, hdim)
    self.Wv = nn.Linear(hdim, hdim)
    self.Wo = nn.Linear(hdim, hdim)
    self.num_heads = num_heads
    self.norm1 = nn.LayerNorm(hdim)
    self.norm2 = nn.LayerNorm(hdim)
    self.FFN = FeedForwardNetwork(hdim)

    assert hdim % num_heads == 0, "Please enter heads such that hdim can be split"

  def forward(self, x, mask=None):
    x_norm = self.norm1(x)
    q, k, v = self.Wq(x_norm), self.Wk(x_norm), self.Wv(x_norm)
    B, T, D = q.shape
    # split q, k, v into heads
    q = q.reshape(B, T, self.num_heads, D // self.num_heads).permute(0, 2, 1, 3)
    k = k.reshape(B, T, self.num_heads, D // self.num_heads).permute(0, 2, 1, 3)
    v = v.reshape(B, T, self.num_heads, D // self.num_heads).permute(0, 2, 1, 3)

    dot_prod = q @ k.transpose(-1, -2)  # B, H, T, Dh @ B, H, Dh, T -> B, H, T, T
    if mask is not None:
      dot_prod = torch.masked_fill(dot_prod, mask, -torch.inf)

    scaled_dot_prod = dot_prod / math.sqrt(D // self.num_heads)

    attention = torch.softmax(scaled_dot_prod, dim=-1)  # B, H, T, T
    attention_out = attention @ v  # B, H, T, Dh

    attention_out = attention_out.permute(0, 2, 1, 3).flatten(-2)  # B, T, 768
    attention_out = self.Wo(attention_out)
    residual = attention_out + x

    return residual + self.FFN(self.norm2(residual))


class Transformer(nn.Module):
  def __init__(self, num_layers=2):
    super().__init__()
    self.layers = [TransformerLayer() for _ in range(num_layers)]
    self.layers = nn.Sequential(*self.layers)

  def forward(self, x):
    return self.layers(x)


class VisionTransformer(nn.Module):
  def __init__(self, in_channels, hdim=768):
    super().__init__()
    self.patch_embed = PatchEmbed(in_channels=3)
    self.pos_embed = PositionalEmbedding()
    self.transformer_blocks = Transformer(num_layers=10)
    self.cls_token = nn.Parameter(torch.zeros(1, 1, hdim))

  def forward(self, x):
    B, C, H, W = x.shape
    x = self.patch_embed(x)

    cls = self.cls_token.expand(B, -1, -1)
    x = torch.concat([cls, x], dim=1)
    x = self.pos_embed(x)
    x = self.transformer_blocks(x)
    return x


if __name__ == "__main__":
  vit = VisionTransformer(3, 768)
  out = vit(torch.randn(4, 3, 256, 256))
  # print(out.shape)  # torch.Size([4, 257, 768])

## Shape check

In [4]:
ViT = VisionTransformer(3, 768)
out = ViT(torch.randn(4, 3, 256, 256))

In [5]:
out.shape

torch.Size([4, 257, 768])